# 02 — LoRA SFT training (Colab T4)

Fine-tune **Llama-3.2-1B-Instruct** with LoRA on JSONL shards from notebook **01**:

- **S3** — MIDI-Instruct edit shards (default)
- **S2** — MidiCaps caption↔MIDI shards (set `STAGE="S2"`; run notebook 01 with `BUILD_MIDICAPS=True` first)

**Self-contained:** bootstraps a fresh Colab runtime (clone, install, restore/build shards). Keep `RUN_ID` and `USE_DRIVE` aligned with notebook **01**.

In [ ]:
# @title Configuration (must match notebook 01)
MIDI_LLM_REPO = "https://github.com/juliagsy/midi-llm.git"
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"

WORK = "/content"
MIDI_LLM = f"{WORK}/midi-llm"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
RUN_ID = "pilot_v1"
RUNS_ROOT = f"{WORK}/runs/{RUN_ID}"
SHARDS_ROOT = f"{WORK}/data/shards/{RUN_ID}"

# Training knobs — T4 defaults
STAGE = "S3"  # @param ["S3", "S2"]  S3=edit (MIDI-Instruct), S2=caption↔MIDI (MidiCaps)
REPR = "remi"  # @param ["remi", "octuple", "amt"]
TRAIN_ALL_REPRS = False  # @param {type:"boolean"}
MAX_STEPS = 50  # @param {type:"integer"}  pilot: 50 (loss ~0.001 @ ~40); full run: 300+
MAX_SEQ_LEN = 1024  # @param {type:"integer"}  # T4 override; protocol default is 2048 (configs/base.yaml)
MAX_SAMPLES = None  # @param {type:"raw"}  # None = full train shard
BATCH_SIZE = 2  # @param {type:"integer"}
GRAD_ACCUM = None  # @param {type:"raw"}  # None → derive from configs/base.yaml effective_batch_tokens
LEARNING_RATE = 2e-4  # @param {type:"number"}
PRECISION = "fp16"  # @param ["fp16", "bf16"]  T4 → fp16
SEED = 42  # @param {type:"integer"}

USE_DRIVE = True  # @param {type:"boolean"}  recommended: each notebook gets a new runtime
DRIVE_ROOT = "/content/drive/MyDrive/midi-llm"
DRIVE_SHARDS = f"{DRIVE_ROOT}/shards/{RUN_ID}"
DRIVE_PILOT = f"{DRIVE_ROOT}/pilot/{RUN_ID}"
DRIVE_MIDICAPS = f"{DRIVE_ROOT}/midicaps"
DRIVE_RUNS = f"{DRIVE_ROOT}/runs/{RUN_ID}"

MIDICAPS_ROOT = f"{WORK}/data/midicaps"
SHARD_PREFIX = "edit" if STAGE == "S3" else "midicaps"

In [ ]:
# @title Mount Google Drive (checkpoints + shard restore)
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = DRIVE_RUNS
    os.makedirs(RUNS_ROOT, exist_ok=True)
    print("Checkpoints on Drive:", RUNS_ROOT)
    print("Shard restore path:", DRIVE_SHARDS)
else:
    print("Ephemeral VM paths — shards:", SHARDS_ROOT, "| runs:", RUNS_ROOT)

In [ ]:
# @title Bootstrap session (clone, install, restore or build shards)
import os
import shutil
import subprocess
from pathlib import Path

def _sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

_sh(f"rm -rf {MIDI_LLM} {MUSICINSTRUCT}")
_sh(f"git clone --branch {BRANCH} --depth 1 {MIDI_LLM_REPO} {MIDI_LLM}")
_sh(f"git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}")

%cd {MIDI_LLM}
# Colab torchao breaks peft; LoRA does not need it
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q -U "transformers>=4.44,<5" "accelerate>=0.33"
!pip install -q -e ".[repr,data,train]"
!pip install -q -e "{MUSICINSTRUCT}"
!pip install -q "git+https://github.com/jthickstun/anticipation.git" mido

if not os.environ.get("HF_TOKEN"):
    from getpass import getpass
    from huggingface_hub import login
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

manifest = Path(MUSICINSTRUCT) / "data/pilot/pilot.jsonl"
shards_dir = Path(SHARDS_ROOT)
shard_ok = shards_dir.is_dir() and any(shards_dir.glob(f"{SHARD_PREFIX}_*_train.jsonl"))

if not shard_ok and USE_DRIVE and Path(DRIVE_SHARDS).is_dir():
    shards_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_SHARDS, SHARDS_ROOT, dirs_exist_ok=True)
    shard_ok = any(shards_dir.glob(f"{SHARD_PREFIX}_*_train.jsonl"))
    if shard_ok:
        print("Restored shards from Drive:", DRIVE_SHARDS)

if not shard_ok:
    print(f"Building {SHARD_PREFIX} shards (first run or no Drive copy)...")
    os.makedirs(SHARDS_ROOT, exist_ok=True)
    if STAGE == "S3":
        pilot_dir = Path(MUSICINSTRUCT) / "data/pilot"
        if USE_DRIVE and Path(DRIVE_PILOT).is_dir():
            pilot_dir.mkdir(parents=True, exist_ok=True)
            shutil.copytree(DRIVE_PILOT, pilot_dir, dirs_exist_ok=True)
            print("Restored pilot from Drive:", DRIVE_PILOT)
        elif not manifest.is_file():
            !cd {MUSICINSTRUCT} && musicinstruct generate-pilot --output-dir data/pilot --target 300
        for split in ("train", "validation"):
            for repr_name in ("remi", "octuple", "amt"):
                out = f"{SHARDS_ROOT}/edit_{repr_name}_{split}.jsonl"
                !midi-llm build-instruct-shard {manifest} --repr {repr_name} --output {out} --split {split}
    else:
        midicaps_root = Path(MIDICAPS_ROOT)
        if USE_DRIVE and Path(DRIVE_MIDICAPS).is_dir():
            midicaps_root.mkdir(parents=True, exist_ok=True)
            shutil.copytree(DRIVE_MIDICAPS, midicaps_root, dirs_exist_ok=True)
            print("Restored MidiCaps from Drive:", DRIVE_MIDICAPS)
        elif not (midicaps_root / "lmd_full").is_dir():
            raise FileNotFoundError(
                "No MidiCaps on Drive. Run notebook 01 with BUILD_MIDICAPS=True and Sync to Drive."
            )
        for repr_name in ("remi", "octuple", "amt"):
            out = f"{SHARDS_ROOT}/midicaps_{repr_name}_train.jsonl"
            !midi-llm build-midicaps-shard --midi-root {MIDICAPS_ROOT} --repr {repr_name} --output {out} --limit 500
    if USE_DRIVE:
        Path(DRIVE_SHARDS).parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(SHARDS_ROOT, DRIVE_SHARDS, dirs_exist_ok=True)
        print("Synced new shards to Drive:", DRIVE_SHARDS)

print("Shard files:", sorted(p.name for p in shards_dir.glob("*.jsonl")))

In [ ]:
# @title GPU check
import torch
assert torch.cuda.is_available(), "Enable T4 GPU runtime"
print(torch.cuda.get_device_name(0))

In [ ]:
# @title Train LoRA
from pathlib import Path
from midi_llm.config import load_config
from midi_llm.config_helpers import resolve_grad_accum_steps
from midi_llm.train.lora_sft import save_lora_artifacts, setup_lora_trainer

reprs = ["remi", "octuple", "amt"] if TRAIN_ALL_REPRS else [REPR]
results = {}
trainer = None
out_dir = None
train_meta = None

for repr_name in reprs:
    shard = Path(SHARDS_ROOT) / f"{SHARD_PREFIX}_{repr_name}_train.jsonl"
    if not shard.is_file():
        raise FileNotFoundError(
            f"Missing shard {shard} — run notebook 01 ({STAGE}) or bootstrap cell above"
        )
    out_dir = Path(RUNS_ROOT) / (f"{repr_name}_s2" if STAGE == "S2" else repr_name)
    cfg = load_config(repr_name)
    protocol_seq = cfg["model"]["max_seq_len"]
    if MAX_SEQ_LEN != protocol_seq:
        print(f"Note: MAX_SEQ_LEN={MAX_SEQ_LEN}; paper protocol is {protocol_seq}")
    grad_accum = GRAD_ACCUM
    if grad_accum is None:
        grad_accum = resolve_grad_accum_steps(
            cfg, seq_len=MAX_SEQ_LEN, batch_size=BATCH_SIZE
        )
    save_steps = min(50, max(10, MAX_STEPS // 6))
    print(f"\n=== Training {repr_name} → {out_dir} (grad_accum={grad_accum}) ===")
    trainer, out_dir, train_meta = setup_lora_trainer(
        shard,
        out_dir,
        repr_name=repr_name,
        max_steps=MAX_STEPS,
        max_seq_len=MAX_SEQ_LEN,
        max_samples=MAX_SAMPLES,
        per_device_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=grad_accum,
        learning_rate=LEARNING_RATE,
        precision=PRECISION,
        seed=SEED,
        logging_steps=10,
        save_steps=save_steps,
    )
    trainer.train()
    adapter = save_lora_artifacts(trainer, out_dir, train_meta)
    results[repr_name] = str(adapter)
    print("Saved:", adapter)

print("\nAdapters:", results)

In [ ]:
# @title Manual save (■ stop Train cell first, then run this)
import shutil
from pathlib import Path
from midi_llm.train.lora_sft import save_lora_artifacts

if trainer is None or out_dir is None or train_meta is None:
    raise RuntimeError(
        "No trainer in memory. Run Train LoRA, click ■ Stop when loss has plateaued, then run this cell."
    )

adapter = save_lora_artifacts(trainer, out_dir, train_meta)
print(f"Saved step {trainer.state.global_step} → {adapter}")

if USE_DRIVE:
    Path(DRIVE_RUNS).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUNS_ROOT, DRIVE_RUNS, dirs_exist_ok=True)
    print("Synced to Drive:", DRIVE_RUNS)

In [ ]:
# @title Sync checkpoints to Drive + optional zip download
import shutil
from pathlib import Path

if USE_DRIVE:
    Path(DRIVE_RUNS).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUNS_ROOT, DRIVE_RUNS, dirs_exist_ok=True)
    print("Synced checkpoints to Drive:", DRIVE_RUNS)

zip_path = f"{WORK}/runs_{RUN_ID}_adapters"
shutil.make_archive(zip_path, "zip", RUNS_ROOT)
print("Local zip:", zip_path + ".zip")
try:
    from google.colab import files
    files.download(zip_path + ".zip")
except ImportError:
    pass

## T4 timing guide

| `MAX_STEPS` | `MAX_SEQ_LEN` | ~Time (1 arm) |
|-------------|---------------|---------------|
| 50 | 512 | 5–10 min |
| 300 | 1024 | 30–45 min |
| 3000 | 1024 | 4–6 h |

If OOM: set `BATCH_SIZE=1`, `MAX_SEQ_LEN=512`, `GRAD_ACCUM=8`.

## Next

1. Confirm **Sync checkpoints to Drive** ran (if `USE_DRIVE=True`).
2. Disconnect this runtime (optional).
3. Open **`03_eval_musicinstruct.ipynb`** in a **new** session (same `RUN_ID`, `USE_DRIVE=True`).